# IRIS Phase 2: build the full report on a Colab GPU

Builds `phase2_results.md` from the cached out-of-fold results of every model (including the DistilRoBERTa results from your first
Colab run), refits the cheaper models once on all development users to score ordinary chat (ESConv) and the challenge set, and runs
the project's own report code unchanged. Nothing is typed in by hand. About 15 to 25 minutes on a T4.

**Before you start:** Runtime > Change runtime type > **T4 GPU**, and put `iris_colab_report_bundle.zip` in the root of your Drive.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/iris_phase2_colab'
import os, zipfile, shutil
os.makedirs(OUT, exist_ok=True)
ZIP = '/content/drive/MyDrive/iris_colab_report_bundle.zip'
if not os.path.exists(ZIP):
    from google.colab import files
    up = files.upload()
    ZIP = next(iter(up))
shutil.rmtree('/content/iris', ignore_errors=True)
zipfile.ZipFile(ZIP).extractall('/content/iris')
%cd /content/iris/ml
!pip install -q -r requirements-colab.txt
!python colab/colab_report.py --out {OUT} --dry-run

## Build the report (prints progress; the false-alarm audit refits each model once)

In [ ]:
!python -u colab/colab_report.py --out {OUT}

## Look at it, then download

In [ ]:
print(open(f'{OUT}/phase2_results.md').read()[:6000])

In [ ]:
from google.colab import files
files.download(f'{OUT}/iris_report_results.zip')